# Query

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import boto3
import os
import logging
import pandas as pd

In [2]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (SparkSession
    .builder
    .appName("FlightDataETLQuery")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())

spark.sparkContext.setLogLevel("FATAL")
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/rwjlb/.ivy2.5.2/cache
The jars for the packages stored in: /home/rwjlb/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a123d47f-0ee5-497d-b529-22815becec8e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
:: resolution report :: resolve 461ms :: artifacts dl 20ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-a

In [3]:
spark_df = spark.read.parquet("s3a://rwc-ml-datasets/raw/Flight_Delay_Prediction_Datasets/combined_flight_data/")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

In [4]:
spark_df.createOrReplaceTempView("flight_data")

In [5]:
spark_df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_AIRLINE_ID: integer (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- ORIGIN_CITY_MARKET_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- ORIGIN_STATE_FIPS: integer (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- ORIGIN_WAC: integer (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)

In [6]:
spark.sql("SELECT * FROM flight_data LIMIT 3").toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,...,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,FIRST_DEP_TIME,TOTAL_ADD_GTIME
0,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8317M,...,1.0,957.0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8317M,...,1.0,387.0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8317M,...,1.0,1984.0,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
origin_airports = spark.sql("SELECT DISTINCT ORIGIN FROM flight_data").toPandas()['ORIGIN']

In [8]:
origin_airports

0      BGM
1      PSE
2      INL
3      DLG
4      MSY
      ... 
347    STT
348    SBP
349    RAP
350    CLD
351    ASE
Name: ORIGIN, Length: 352, dtype: str

In [9]:
dest_airports = spark.sql("SELECT DISTINCT DEST FROM flight_data").toPandas()['DEST']

In [10]:
dest_airports

0      BGM
1      PSE
2      INL
3      DLG
4      MSY
      ... 
347    STT
348    SBP
349    RAP
350    CLD
351    ASE
Name: DEST, Length: 352, dtype: str

In [11]:
airports = pd.concat([origin_airports, dest_airports], ignore_index=True)

In [12]:
airports

0      BGM
1      PSE
2      INL
3      DLG
4      MSY
      ... 
699    STT
700    SBP
701    RAP
702    CLD
703    ASE
Length: 704, dtype: str

In [13]:
airports = airports.drop_duplicates()

In [14]:
airports

0      BGM
1      PSE
2      INL
3      DLG
4      MSY
      ... 
347    STT
348    SBP
349    RAP
350    CLD
351    ASE
Length: 352, dtype: str

## Export the airport list to a JSON list

In [15]:
import json

In [16]:
airports_json = json.dumps(airports.tolist())

In [17]:
s3_bucket = 'rwc-ml-datasets'
s3_path = 'raw/Flight_Delay_Prediction_Datasets/other_data/airports.json'

In [18]:
s3_client = boto3.client('s3')

In [19]:
s3_client.put_object(
    Bucket=s3_bucket,
    Key=s3_path,
    Body=airports_json,
    ContentType='application/json'
)

{'ResponseMetadata': {'RequestId': 'VXEY643XXCKYDA28',
  'HostId': 'sxf8AYMvaXzUJK0pZC7K8/aSyFQpCoVueEY4O28pSqdpf2vcTjfVc1O882hQpb/8QHwkc0uG7DM7ECbAQ4/TWJ8E/7Y3ItGD',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'sxf8AYMvaXzUJK0pZC7K8/aSyFQpCoVueEY4O28pSqdpf2vcTjfVc1O882hQpb/8QHwkc0uG7DM7ECbAQ4/TWJ8E/7Y3ItGD',
   'x-amz-request-id': 'VXEY643XXCKYDA28',
   'date': 'Thu, 02 Jul 2026 17:19:23 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"6ede9238adb168792ad85b7afc749852"',
   'x-amz-checksum-crc32': 'q3bzKg==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 1},
 'ETag': '"6ede9238adb168792ad85b7afc749852"',
 'ChecksumCRC32': 'q3bzKg==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}